In [ ]:
import os
# macOS fix: FAISS and PyTorch each ship their own OpenMP runtime; loading both in one
# process crashes the kernel on CPU. These two settings must be set BEFORE importing them.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import re, time, warnings
import numpy as np
import pandas as pd
import requests
import faiss
import pypdf
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
import torch; torch.set_num_threads(1)   # keep PyTorch single-threaded (see OpenMP note above)

# ---------------- Configuration ----------------
PDF_PATH        = "data/Bharatiya_Nyaya_Sanhita_2023.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"   # Bi-Encoder
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # Cross-Encoder
OLLAMA_URL      = "http://localhost:11434/api/generate"
LLM_MODEL       = "llama3"
DEVICE          = "cpu"   # encoders run on CPU; the Mac GPU (Metal) is left free for the Ollama LLM

CHUNK_WORDS     = 150   # words per chunk
OVERLAP_WORDS   = 30    # overlap between consecutive chunks (so a clause is not cut in half)
TOP_K_RETRIEVE  = 10    # candidates fetched by the bi-encoder
TOP_K_CONTEXT   = 3     # chunks finally given to the LLM

In [ ]:
reader = pypdf.PdfReader(PDF_PATH)
print("Pages in PDF:", len(reader.pages))

# Body text lies roughly between x=245 and x=900 (in pypdf's transformed units);
# margin notes sit at x≈120 (even pages) or x≈1010 (odd pages).
MARGIN_LEFT, MARGIN_RIGHT = 200, 980

def extract_body_text(page):
    # Return only the body-column text of a page, rebuilding lines from y-coordinates.
    lines, current, last_y = [], [], None

    def visitor(text, cm, tm, font_dict, font_size):
        nonlocal current, last_y
        if not text.strip():                 # keep whitespace inside a line
            if text and current:
                current.append(text)
            return
        x = tm[4] * cm[0] + cm[4]
        y = tm[5] * cm[3] + cm[5]
        if x < MARGIN_LEFT or x > MARGIN_RIGHT:   # skip margin-note column
            return
        if last_y is not None and abs(y - last_y) > 3:   # new line started
            lines.append("".join(current)); current = []
        current.append(text); last_y = y

    page.extract_text(visitor_text=visitor)
    lines.append("".join(current))
    return "\n".join(l.strip() for l in lines)

FOOTER_MARKERS = ("DIW AKAR SINGH", "Joint Secretary", "UPLOADED BY", "MGIPMRND",
                  "PUBLISHED BY THE CONTROLLER")

def clean_page(text, page_no):
    # Remove headers, the page-1 masthead and the last-page signature block.
    if page_no == 1 and "vlk/kkj" in text:       # Hindi masthead starts here
        text = text[:text.index("vlk/kkj")]
    kept = []
    for line in text.split("\n"):
        s = line.strip()
        if not s or "GAZETTE OF INDIA" in s or re.fullmatch(r"[_\s]+", s):
            continue
        if s.startswith("— —") or any(m in s for m in FOOTER_MARKERS):
            break                                  # end of the Act's text
        kept.append(s)
    return "\n".join(kept)

pages = [clean_page(extract_body_text(p), i + 1) for i, p in enumerate(reader.pages)]
print("Total words after cleaning:", sum(len(p.split()) for p in pages))
print("\n--- Sample (page 12) ---\n", pages[11][:700])

In [ ]:
reader = pypdf.PdfReader(PDF_PATH)
print("Pages in PDF:", len(reader.pages))

# Body text lies roughly between x=245 and x=900 (in pypdf's transformed units);
# margin notes sit at x≈120 (even pages) or x≈1010 (odd pages).
MARGIN_LEFT, MARGIN_RIGHT = 200, 980

def extract_body_text(page):
    # Return only the body-column text of a page, rebuilding lines from y-coordinates.
    lines, current, last_y = [], [], None

    def visitor(text, cm, tm, font_dict, font_size):
        nonlocal current, last_y
        if not text.strip():                 # keep whitespace inside a line
            if text and current:
                current.append(text)
            return
        x = tm[4] * cm[0] + cm[4]
        y = tm[5] * cm[3] + cm[5]
        if x < MARGIN_LEFT or x > MARGIN_RIGHT:   # skip margin-note column
            return
        if last_y is not None and abs(y - last_y) > 3:   # new line started
            lines.append("".join(current)); current = []
        current.append(text); last_y = y

    page.extract_text(visitor_text=visitor)
    lines.append("".join(current))
    return "\n".join(l.strip() for l in lines)

FOOTER_MARKERS = ("DIW AKAR SINGH", "Joint Secretary", "UPLOADED BY", "MGIPMRND",
                  "PUBLISHED BY THE CONTROLLER")

def clean_page(text, page_no):
    # Remove headers, the page-1 masthead and the last-page signature block.
    if page_no == 1 and "vlk/kkj" in text:       # Hindi masthead starts here
        text = text[:text.index("vlk/kkj")]
    kept = []
    for line in text.split("\n"):
        s = line.strip()
        if not s or "GAZETTE OF INDIA" in s or re.fullmatch(r"[_\s]+", s):
            continue
        if s.startswith("— —") or any(m in s for m in FOOTER_MARKERS):
            break                                  # end of the Act's text
        kept.append(s)
    return "\n".join(kept)

pages = [clean_page(extract_body_text(p), i + 1) for i, p in enumerate(reader.pages)]
print("Total words after cleaning:", sum(len(p.split()) for p in pages))
print("\n--- Sample (page 12) ---\n", pages[11][:700])

In [ ]:
# 1) Flatten the document into words, remembering the page and section of every word.
words, section = [], 0
for page_no, text in enumerate(pages, start=1):
    for line in text.split("\n"):
        # A new section starts with "<n>." followed by text, e.g. "101.Except in the cases..."
        m = re.match(r"^(\d{1,3})\.\s?(\(1\))?\s?[A-Z“(]", line)
        if m and section < int(m.group(1)) <= section + 2:   # sections appear in increasing order
            section = int(m.group(1))
        for w in line.split():
            words.append((w, page_no, section))

# 2) Sliding window over the words.
chunks = []
step = CHUNK_WORDS - OVERLAP_WORDS
for start in range(0, len(words), step):
    seg = words[start:start + CHUNK_WORDS]
    first, last = seg[0][2], seg[-1][2]
    chunks.append({
        "chunk_id": len(chunks),
        "page": seg[0][1],
        "section": f"Sec {first}" if first == last else f"Sec {first}-{last}",
        "text": " ".join(w for w, _, _ in seg),
    })
    if start + CHUNK_WORDS >= len(words):
        break

print(f"Sections detected: {section} | Chunks created: {len(chunks)}")
pd.DataFrame(chunks)[["chunk_id", "page", "section", "text"]].head(3)